# 13. Binäre Suche: halbieren statt durchprobieren

Am Ende von [Notebook 12](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/12_greedy.ipynb) stand ein Ausblick: Man **rät einen Zielwert** und prüft nur
„Geht es mit höchstens `x`?“, ein gieriger Test genügt oft. Wenn die Antwort für große `x` „ja“ und für kleine „nein“ ist (oder
umgekehrt), muss man nicht alle `x` durchprobieren. Man schaut in die **Mitte** und wirft damit jedes Mal die **Hälfte** der Möglichkeiten weg.
Das ist die **binäre Suche**. In Notebook 1 hast du Durchprobieren kennengelernt, hier siehst du, wie man oft sehr viel weniger probieren muss.

In diesem Notebook lernst du:

- die **binäre Suche** in einer sortierten Liste selbst zu schreiben und zu zählen, wie viele Schritte sie braucht (**O(log n)**),
- mit dem Modul `bisect` die Einfügestelle zu finden und Elemente in einem Bereich zu zählen,
- typische **Fehler** zu erkennen: Suche in einer unsortierten Liste, Endlosschleife durch `links = mitte`,
- einen **Zielwert zu raten und zu prüfen**: aus „Geht es mit höchstens `x`?“ wird die kleinste Zahl `x`, für die es geht,
- **Meet in the Middle**: Man teilt die Möglichkeiten in zwei Hälften, berechnet für jede Hälfte alle Summen, sortiert eine davon und schlägt
  die Partner nach. Statt 3ⁿ Möglichkeiten sind es nur etwa 2 · 3ⁿᐟ².

In der Wettbewerbs-Ecke wählst du Hotels so, dass die niedrigste Bewertung möglichst hoch ist, und für Schnelle wiegst du mit einer Marktwaage.

## 1. Zahl raten: halbieren

Ich denke mir eine Zahl zwischen 1 und 100. Du darfst Fragen der Form „Ist deine Zahl größer als 50?“ stellen, ich antworte nur mit ja oder nein.
Wer es klug macht, fragt immer nach der **Mitte** des Bereichs, der noch übrig ist, und halbiert ihn so.

#### Frage 1.1

Nimm Stift und Papier. Die gesuchte Zahl ist 73, der Bereich am Anfang 1 bis 100. Frage immer nach der Mitte („Ist sie größer als 50?“, bei einem
Bereich mit ungerader Länge rundest du die Mitte ab) und schreibe jedes Mal den Bereich auf, der noch übrig ist.

1. Wie viele Fragen brauchst du, bis nur noch eine Zahl übrig ist?
2. Wie viele Fragen braucht man **höchstens** bei den Zahlen von 1 bis 100, von 1 bis 1 000, von 1 bis 1 000 000?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Bereiche: 1 bis 100, Frage „größer als 50?“, ja: 51 bis 100; „größer als 75?“, nein: 51 bis 75; „größer als 63?“, ja: 64 bis 75; „größer als 69?“, ja: 70 bis 75;
   „größer als 72?“, ja: 73 bis 75; „größer als 74?“, nein: 73 bis 74; „größer als 73?“, nein: 73. Das sind **7 Fragen**.
2. Der Bereich wird mit jeder Frage etwa halb so groß. Mit 7 Fragen schafft man 2⁷ = 128 Zahlen, mit 10 Fragen 2¹⁰ = 1 024 Zahlen, mit 20 Fragen 2²⁰ = 1 048 576
   Zahlen. Also **höchstens 7, 10 und 20 Fragen**. Eine Million Zahlen sind also nach 20 Fragen erledigt. Wer der Reihe nach „Ist es 1? Ist es 2? …“ fragt, braucht im
   schlechtesten Fall 1 000 000.

</details>

### Binäre Suche in einer sortierten Liste

Dasselbe Spiel in einer **sortierten** Liste: Man sucht den Index eines Werts `ziel`. Wir merken uns zwei Grenzen `links` und `rechts`: Wenn `ziel` in der Liste
steht, dann zwischen diesen beiden Indizes (einschließlich). Am Anfang sind das `0` und `len(liste) - 1`. In jeder Runde schaut man in die Mitte `mitte = (links + rechts) // 2`:

- ist `liste[mitte] == ziel`: gefunden,
- ist `liste[mitte] < ziel`: `ziel` steht rechts davon, also `links = mitte + 1`,
- sonst steht `ziel` links davon: `rechts = mitte - 1`.

Wird `links > rechts`, ist der Bereich leer: Das Ziel steht nicht in der Liste. Zum Vergleich schreiben wir zuerst die **lineare Suche** (der Reihe nach
ansehen, wie beim Durchprobieren) und zählen bei beiden die Schritte, also wie oft ein Element angesehen wird.

#### Aufgabe 1.2

Schreibe `suche_linear(liste, ziel)`: Sie geht die Liste der Reihe nach durch und gibt ein Paar `(index, schritte)` zurück. `schritte` zählt, wie viele Elemente angesehen wurden.
Steht `ziel` nicht in der Liste, ist `index` gleich `-1`.

**Erwartete Ausgabe:**
```
47 (9, 10)
3 (0, 1)
77 (14, 15)
50 (-1, 15)
```

In [ ]:
zahlen = [3, 8, 12, 17, 21, 25, 31, 36, 42, 47, 53, 58, 64, 70, 77]

def suche_linear(liste, ziel):
    schritte = 0
    # dein Code
    return -1, schritte

for ziel in [47, 3, 77, 50]:
    print(ziel, suche_linear(zahlen, ziel))

<details>
<summary>Tipp anzeigen</summary>

`for i in range(len(liste)):` Zähle zuerst `schritte` hoch und vergleiche dann `liste[i] == ziel`. Bei einem Treffer gibst du `i, schritte` zurück, nach der Schleife `-1, schritte`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
zahlen = [3, 8, 12, 17, 21, 25, 31, 36, 42, 47, 53, 58, 64, 70, 77]

def suche_linear(liste, ziel):
    schritte = 0
    for i in range(len(liste)):
        schritte = schritte + 1
        if liste[i] == ziel:
            return i, schritte
    return -1, schritte

for ziel in [47, 3, 77, 50]:
    print(ziel, suche_linear(zahlen, ziel))
```

Die 47 steht an Index 9, das ist das 10. Element, also 10 Schritte. Die 3 steht vorn (1 Schritt), die 77 ganz hinten (15 Schritte). Die 50 fehlt: Man muss
**alle 15** ansehen, bevor man sicher ist.

</details>

#### Aufgabe 1.3

Schreibe `suche_binaer(liste, ziel)` nach der Beschreibung oben. Auch sie gibt `(index, schritte)` zurück, `schritte` zählt, wie oft du in die Mitte geschaut hast.

**Erwartete Ausgabe:**
```
47 (9, 3)
3 (0, 4)
77 (14, 4)
50 (-1, 4)
```

In [ ]:
zahlen = [3, 8, 12, 17, 21, 25, 31, 36, 42, 47, 53, 58, 64, 70, 77]

def suche_binaer(liste, ziel):
    links = 0
    rechts = len(liste) - 1
    schritte = 0
    # dein Code
    return -1, schritte

for ziel in [47, 3, 77, 50]:
    print(ziel, suche_binaer(zahlen, ziel))

<details>
<summary>Tipp anzeigen</summary>

`while links <= rechts:` Darin `mitte = (links + rechts) // 2` und `schritte` erhöhen. Dann die drei Fälle: gleich (zurückgeben), kleiner (`links = mitte + 1`),
größer (`rechts = mitte - 1`).

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
zahlen = [3, 8, 12, 17, 21, 25, 31, 36, 42, 47, 53, 58, 64, 70, 77]

def suche_binaer(liste, ziel):
    links = 0
    rechts = len(liste) - 1
    schritte = 0
    while links <= rechts:
        mitte = (links + rechts) // 2
        schritte = schritte + 1
        if liste[mitte] == ziel:
            return mitte, schritte
        if liste[mitte] < ziel:
            links = mitte + 1
        else:
            rechts = mitte - 1
    return -1, schritte

for ziel in [47, 3, 77, 50]:
    print(ziel, suche_binaer(zahlen, ziel))
```

Die 47: `mitte` ist zuerst 7 (Wert 36, zu klein), dann 11 (58, zu groß), dann 9: gefunden nach **3** Schritten (linear waren es 10). Die 3 und die 77 am Rand brauchen 4 Schritte.
Die fehlende 50 braucht ebenfalls 4 Schritte, dann ist `links > rechts` und man weiß: nicht dabei. Bei 15 Elementen sind das nie mehr als 4 Schritte.

</details>

#### Aufgabe 1.4

Wie viele Schritte braucht man **im schlechtesten Fall**? Schreibe `schlimmster_fall(n)`. Sie bildet `liste = list(range(n))` und gibt ein Paar `(linear, binaer)` zurück:
`linear` ist die Schrittzahl der linearen Suche nach dem **letzten** Element (das ist bei der linearen Suche der schlechteste Fall), `binaer` ist das **Maximum** der
Schrittzahlen der binären Suche über **alle** Elemente der Liste als Ziel.

**Erwartete Ausgabe:**
```
10 (10, 4)
100 (100, 7)
1000 (1000, 10)
10000 (10000, 14)
```

In [ ]:
def suche_linear(liste, ziel):
    schritte = 0
    for i in range(len(liste)):
        schritte = schritte + 1
        if liste[i] == ziel:
            return i, schritte
    return -1, schritte

def suche_binaer(liste, ziel):
    links = 0
    rechts = len(liste) - 1
    schritte = 0
    while links <= rechts:
        mitte = (links + rechts) // 2
        schritte = schritte + 1
        if liste[mitte] == ziel:
            return mitte, schritte
        if liste[mitte] < ziel:
            links = mitte + 1
        else:
            rechts = mitte - 1
    return -1, schritte

def schlimmster_fall(n):
    liste = list(range(n))
    linear = 0
    binaer = 0
    # dein Code
    return linear, binaer

for n in [10, 100, 1000, 10000]:
    print(n, schlimmster_fall(n))

<details>
<summary>Tipp anzeigen</summary>

`linear = suche_linear(liste, n - 1)[1]`. Für `binaer`: Schleife über alle `ziel in liste`, merke das größte `suche_binaer(liste, ziel)[1]`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def suche_linear(liste, ziel):
    schritte = 0
    for i in range(len(liste)):
        schritte = schritte + 1
        if liste[i] == ziel:
            return i, schritte
    return -1, schritte

def suche_binaer(liste, ziel):
    links = 0
    rechts = len(liste) - 1
    schritte = 0
    while links <= rechts:
        mitte = (links + rechts) // 2
        schritte = schritte + 1
        if liste[mitte] == ziel:
            return mitte, schritte
        if liste[mitte] < ziel:
            links = mitte + 1
        else:
            rechts = mitte - 1
    return -1, schritte

def schlimmster_fall(n):
    liste = list(range(n))
    linear = suche_linear(liste, n - 1)[1]
    binaer = 0
    for ziel in liste:
        schritte = suche_binaer(liste, ziel)[1]
        if schritte > binaer:
            binaer = schritte
    return linear, binaer

for n in [10, 100, 1000, 10000]:
    print(n, schlimmster_fall(n))
```

Bei der linearen Suche wächst die Schrittzahl mit `n` (10, 100, 1000, 10000): Das ist **O(n)**. Bei der binären Suche sind es 4, 7, 10 und 14 Schritte. Wird die Liste **zehnmal so
lang**, kommen nur etwa 3 bis 4 Schritte dazu. Das ist **O(log n)**.

</details>

#### Frage 1.5

1. Warum braucht die binäre Suche bei `n` Elementen nur etwa log₂(n) Schritte? Wie viele Elemente sind nach 1, 2, 3 Schritten höchstens noch im Suchbereich?
2. Ungefähr wie viele Schritte braucht sie für eine Liste mit einer Million Elementen? Und mit einer Milliarde?
3. Warum funktioniert die binäre Suche nur, wenn die Liste **sortiert** ist? Was passiert in der Rechnung oben, wenn nicht?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Nach jedem Schritt ist der Suchbereich höchstens **halb so groß** (das Element in der Mitte und die eine Hälfte fallen weg): nach 1 Schritt höchstens n/2, nach 2 Schritten n/4,
   nach 3 Schritten n/8, nach `k` Schritten n/2ᵏ. Spätestens wenn 2ᵏ größer als n ist, ist nichts mehr übrig. Das gilt für `k` ≈ log₂(n). Also **O(log n)** Schritte.
2. 2²⁰ ist etwa eine Million, also etwa 20 Schritte. 2³⁰ ist etwa eine Milliarde, also nur etwa 30 Schritte. Die lineare Suche braucht dann eine Million oder eine Milliarde.
3. Der Schluss „ist `liste[mitte] < ziel`, dann steht `ziel` rechts davon“ stimmt nur, wenn alle Elemente links von `mitte` kleiner oder gleich sind. In einer unsortierten Liste
   schmeißt man damit Elemente weg, die das Ziel enthalten könnten (siehe Abschnitt 2). Man muss die Liste also vorher sortieren (das kostet einmalig O(n log n)).
   Das lohnt sich, wenn man **oft** sucht.

</details>

### Das Modul `bisect`

Die binäre Suche braucht man so oft, dass Python sie schon fertig hat: das Modul `bisect`. Die Funktion `bisect_left(liste, x)` gibt die Stelle zurück, an der `x`
in die sortierte Liste **einsortiert** würde, und zwar vor gleichen Elementen. `bisect_right(liste, x)` macht dasselbe **hinter** gleichen Elementen. Beide brauchen nur O(log n) Schritte.
Probiere es aus:

In [ ]:
from bisect import bisect_left, bisect_right
zahlen = [3, 8, 12, 17, 21, 25, 31, 36, 42, 47, 53, 58, 64, 70, 77]
print(bisect_left(zahlen, 47), bisect_right(zahlen, 47))   # 47 steht in der Liste
print(bisect_left(zahlen, 50), bisect_right(zahlen, 50))   # 50 nicht: beide Stellen sind gleich
print(bisect_left(zahlen, 1), bisect_left(zahlen, 100))    # ganz vorn und ganz hinten
doppelt = [1, 2, 2, 2, 5]
print(bisect_left(doppelt, 2), bisect_right(doppelt, 2))   # vor und hinter den drei Zweien

Bei 47 ist `bisect_left` gleich 9 (der Index) und `bisect_right` gleich 10 (dahinter). Bei 50 sind beide 10: Dort würde die 50 einsortiert, vor der 53. Bei 1 kommt 0, bei 100
kommt 15 (die Länge). Bei `[1, 2, 2, 2, 5]` ist die linke Stelle 1 und die rechte 4: Die Differenz `4 - 1 = 3` ist die **Anzahl der Zweien**. Daraus lassen sich zwei
nützliche Funktionen bauen.

#### Aufgabe 1.6

Schreibe mit `bisect_left` und `bisect_right` zwei Funktionen für eine **sortierte** Liste:

- `kommt_vor(liste, x)`: `True`, wenn `x` in der Liste steht. Tipp: Stelle `i = bisect_left(liste, x)`. Wann steht dort `x`? Achtung auf `i == len(liste)`.
- `anzahl_zwischen(liste, a, b)`: Wie viele Elemente `e` haben `a <= e <= b`?

**Erwartete Ausgabe:**
```
True False
6 0 3
```

In [ ]:
zahlen = [3, 8, 12, 17, 21, 25, 31, 36, 42, 47, 53, 58, 64, 70, 77]

from bisect import bisect_left, bisect_right

def kommt_vor(liste, x):
    # dein Code
    return False

def anzahl_zwischen(liste, a, b):
    # dein Code
    return 0

print(kommt_vor(zahlen, 47), kommt_vor(zahlen, 50))
print(anzahl_zwischen(zahlen, 20, 50), anzahl_zwischen(zahlen, 48, 52), anzahl_zwischen([1, 2, 2, 2, 5], 2, 2))

<details>
<summary>Tipp anzeigen</summary>

Bei `kommt_vor`: `i < len(liste) and liste[i] == x`. Bei `anzahl_zwischen`: hinter `b` (`bisect_right`) minus vor `a` (`bisect_left`).

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
zahlen = [3, 8, 12, 17, 21, 25, 31, 36, 42, 47, 53, 58, 64, 70, 77]

from bisect import bisect_left, bisect_right

def kommt_vor(liste, x):
    i = bisect_left(liste, x)
    return i < len(liste) and liste[i] == x          # erst prüfen, ob i noch in der Liste liegt

def anzahl_zwischen(liste, a, b):
    return bisect_right(liste, b) - bisect_left(liste, a)

print(kommt_vor(zahlen, 47), kommt_vor(zahlen, 50))
print(anzahl_zwischen(zahlen, 20, 50), anzahl_zwischen(zahlen, 48, 52), anzahl_zwischen([1, 2, 2, 2, 5], 2, 2))
```

Steht `x` in der Liste, dann an der Stelle `bisect_left`. Ist `i` gleich `len(liste)`, wäre `liste[i]` ein Fehler: Das `and` wertet die rechte Seite nur aus, wenn die linke wahr ist, es prüft das also zuerst. Zwischen 20 und 50 liegen
21, 25, 31, 36, 42 und 47 (6 Stück), zwischen 48 und 52 keine, und bei `[1, 2, 2, 2, 5]` gibt es drei Zweien. Beide Funktionen brauchen nur **O(log n)** Schritte,
selbst wenn es eine Million Elemente gibt.

</details>

## 2. Finde den Fehler

Die binäre Suche ist berüchtigt: Kleine Fehler bei den Grenzen führen zu falschen Ergebnissen oder zu Endlosschleifen.

#### Aufgabe 2.1: Was wird ausgegeben?

Überlege zuerst, was der Code ausgibt, und schreibe es auf. Führe die Zelle erst danach aus.

In [ ]:
def suche(liste, ziel):
    links = 0
    rechts = len(liste) - 1
    while links <= rechts:
        mitte = (links + rechts) // 2
        if liste[mitte] == ziel:
            return mitte
        if liste[mitte] < ziel:
            links = mitte + 1
        else:
            rechts = mitte - 1
    return -1

zahlen = [8, 3, 12, 5, 9]          # nicht sortiert!
print(suche(zahlen, 3), suche(zahlen, 9), suche(zahlen, 12))
print(3 in zahlen, 9 in zahlen)

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

**Ausgabe:**
```
-1 -1 2
True True
```

Ausgegeben werden `-1 -1 2` und `True True`. Die Zahlen 3 und 9 stehen in der Liste, die Suche findet sie aber nicht. Bei der 3: `mitte` ist 2 (der Wert 12 ist größer), also wird
`rechts = 1` gesetzt und die Suche läuft nur noch links, bei `mitte = 0` (8, größer) wird `rechts = -1`: nichts gefunden. Die 3 steht aber an Index 1, die Suche hat sie
übersprungen. Die 12 in der Mitte findet man zufällig. Die binäre Suche **setzt eine sortierte Liste voraus**. Sonst wirft sie mit jeder Entscheidung Elemente weg, die
das Ziel sein könnten, und meldet das **ohne Fehlermeldung**.

</details>

#### Aufgabe 2.2: Was wird ausgegeben?

Überlege zuerst, was der Code ausgibt, und schreibe es auf. Führe die Zelle erst danach aus.

In [ ]:
def suche_falsch(liste, ziel):
    links = 0
    rechts = len(liste) - 1
    runden = 0
    while links <= rechts and runden < 5:      # Sicherung, damit die Zelle nicht ewig läuft
        mitte = (links + rechts) // 2
        runden = runden + 1
        print('Runde', runden, ': links =', links, ' rechts =', rechts, ' mitte =', mitte)
        if liste[mitte] == ziel:
            return mitte
        if liste[mitte] < ziel:
            links = mitte                       # Fehler?
        else:
            rechts = mitte - 1
    return -1

print(suche_falsch([2, 4, 6, 8], 8))

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

**Ausgabe:**
```
Runde 1 : links = 0  rechts = 3  mitte = 1
Runde 2 : links = 1  rechts = 3  mitte = 2
Runde 3 : links = 2  rechts = 3  mitte = 2
Runde 4 : links = 2  rechts = 3  mitte = 2
Runde 5 : links = 2  rechts = 3  mitte = 2
-1
```

In jeder Runde ab der dritten kommt `links = 2`, `rechts = 3` und `mitte = 2` heraus: Der Wert 6 ist kleiner als 8, `links = mitte` ändert **nichts** (links bleibt 2).
Die Schleife läuft immer weiter, ohne dass der Bereich kleiner wird: eine **Endlosschleife**. Ohne die Sicherung `runden < 5` würde die Zelle ewig laufen. Richtig ist
`links = mitte + 1`: Die Mitte wurde schon angesehen und ist nicht das Ziel, darf also aus dem Bereich fallen. Jede Runde muss den Bereich **echt verkleinern**.
Beim Wert `-1` am Ende ist es nur die Sicherung, die die Schleife abgebrochen hat.

</details>

#### Frage 2.3

Finde den Fehler. Lara schreibt die binäre Suche mit `rechts = len(liste)` statt `len(liste) - 1`, sonst bleibt alles gleich:

```python
def suche(liste, ziel):
    links = 0
    rechts = len(liste)
    while links <= rechts:
        mitte = (links + rechts) // 2
        if liste[mitte] == ziel:
            return mitte
        if liste[mitte] < ziel:
            links = mitte + 1
        else:
            rechts = mitte - 1
    return -1

print(suche([2, 4, 6, 8], 4))
print(suche([2, 4, 6, 8], 10))
```

Der erste Aufruf klappt, beim zweiten meldet Python `IndexError: list index out of range`. Woran liegt das, und was ändert sie?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

`rechts` ist der **letzte gültige Index**, also `len(liste) - 1`. Mit `len(liste)` zeigt `rechts` hinter die Liste. Sucht man ein Ziel, das größer ist als alle Elemente, wandert `links` nach
rechts, und irgendwann ist `mitte` gleich `len(liste)`: Der Zugriff `liste[mitte]` geht ins Leere. Der erste Aufruf klappt, weil das Ziel gefunden wird, bevor `mitte` zu groß wird.
Lara muss `rechts = len(liste) - 1` schreiben. (Man kann auch mit `rechts = len(liste)` und `while links < rechts` arbeiten, dann gilt `rechts = mitte` statt `mitte - 1`.
Entscheide dich für **eine** Schreibweise und bleibe dabei.)

</details>

## 3. Zielwert raten und prüfen

Oft sucht man gar nicht ein Element in einer Liste, sondern den **besten Wert** `x` einer Aufgabe, zum Beispiel „den kleinsten möglichen Höchstwert“. Direkt auszurechnen ist das oft
schwer. Einfacher ist die Frage:

> **Geht es mit höchstens `x`?** (ja oder nein)

Diese Frage beantwortet oft ein einfaches **gieriges** Verfahren (Notebook 12). Die Antwort ist **monoton**: Geht es mit `x`, dann auch mit jedem größeren `x`. Dann gibt es eine
**Grenze**: Links davon steht überall „nein“, rechts davon überall „ja“. Genau wie in der sortierten Liste kann man sie durch Halbieren finden.

**Beispiel:** 20 Arbeitspakete mit festen Aufwänden müssen in der **gegebenen Reihenfolge** auf `k` Personen verteilt werden, jede Person bekommt **aufeinanderfolgende** Pakete. Wir wollen,
dass die Person mit der meisten Arbeit möglichst **wenig** zu tun hat. Der kleinste mögliche Höchstwert `x` ist gesucht.

#### Aufgabe 3.1

Schreibe `geht(arbeiten, k, x)`: Sie gibt `True` zurück, wenn man die Pakete in der gegebenen Reihenfolge so auf **höchstens `k`** Personen verteilen kann, dass **keine** Person mehr als
`x` Aufwand hat. Gieriges Verfahren: Die erste Person bekommt so viele Pakete wie möglich, bis das nächste nicht mehr passt; dann ist die nächste Person dran, und so weiter. Ist
ein einzelnes Paket größer als `x`, geht es nie.

**Erwartete Ausgabe:**
```
True False
False True
False True
```

In [ ]:
ARBEITEN = [7, 2, 5, 10, 8, 3, 9, 4, 6, 1, 12, 5, 8, 2, 7, 9, 3, 11, 4, 6]     # Summe 122

def geht(arbeiten, k, x):
    # dein Code
    return False

print(geht(ARBEITEN, 4, 34), geht(ARBEITEN, 4, 33))
print(geht(ARBEITEN, 2, 66), geht(ARBEITEN, 2, 67))
print(geht(ARBEITEN, 1, 121), geht(ARBEITEN, 1, 122))

<details>
<summary>Tipp anzeigen</summary>

Zähle `personen` (am Anfang 1) und die `summe` der aktuellen Person. Für jedes `paket`: Passt `summe + paket > x`, beginnt eine neue Person mit `summe = paket`. Sonst `summe` erhöhen.
Am Ende: `personen <= k`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
ARBEITEN = [7, 2, 5, 10, 8, 3, 9, 4, 6, 1, 12, 5, 8, 2, 7, 9, 3, 11, 4, 6]     # Summe 122

def geht(arbeiten, k, x):
    personen = 1                            # die erste Person
    summe = 0                               # ihr bisheriger Aufwand
    for paket in arbeiten:
        if paket > x:
            return False                    # passt zu keiner Person
        if summe + paket > x:               # passt nicht mehr: nächste Person
            personen = personen + 1
            summe = paket
        else:
            summe = summe + paket
    return personen <= k

print(geht(ARBEITEN, 4, 34), geht(ARBEITEN, 4, 33))
print(geht(ARBEITEN, 2, 66), geht(ARBEITEN, 2, 67))
print(geht(ARBEITEN, 1, 121), geht(ARBEITEN, 1, 122))
```

Alle Pakete zusammen wiegen 122. Mit vier Personen geht es mit `x = 34` gerade noch, mit 33 nicht mehr. Mit zwei Personen braucht man 67, mit einer natürlich alles, also 122.
Wichtig: Die Antwort ist **monoton**. Geht es mit 34, geht es auch mit 35, 36 und so weiter.

</details>

#### Aufgabe 3.2

Schreibe `kleinstes_x_linear(arbeiten, k)`: Sie probiert `x = 1, 2, 3, ...` der Reihe nach, bis `geht` zum ersten Mal `True` liefert. Sie gibt `(x, pruefungen)` zurück, wobei `pruefungen` zählt,
wie oft `geht` aufgerufen wurde.

**Erwartete Ausgabe:**
```
1 (122, 122)
2 (67, 67)
3 (44, 44)
4 (34, 34)
5 (26, 26)
20 (12, 12)
```

In [ ]:
ARBEITEN = [7, 2, 5, 10, 8, 3, 9, 4, 6, 1, 12, 5, 8, 2, 7, 9, 3, 11, 4, 6]     # Summe 122

def geht(arbeiten, k, x):
    personen = 1
    summe = 0
    for paket in arbeiten:
        if paket > x:
            return False
        if summe + paket > x:
            personen = personen + 1
            summe = paket
        else:
            summe = summe + paket
    return personen <= k

def kleinstes_x_linear(arbeiten, k):
    x = 0
    pruefungen = 0
    # dein Code
    return x, pruefungen

for k in [1, 2, 3, 4, 5, 20]:
    print(k, kleinstes_x_linear(ARBEITEN, k))

<details>
<summary>Tipp anzeigen</summary>

`while True:` Erhöhe `x` um 1 und `pruefungen` um 1; wenn `geht(arbeiten, k, x)`, gib `x, pruefungen` zurück.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
ARBEITEN = [7, 2, 5, 10, 8, 3, 9, 4, 6, 1, 12, 5, 8, 2, 7, 9, 3, 11, 4, 6]     # Summe 122

def geht(arbeiten, k, x):
    personen = 1
    summe = 0
    for paket in arbeiten:
        if paket > x:
            return False
        if summe + paket > x:
            personen = personen + 1
            summe = paket
        else:
            summe = summe + paket
    return personen <= k

def kleinstes_x_linear(arbeiten, k):
    x = 0
    pruefungen = 0
    while True:
        x = x + 1
        pruefungen = pruefungen + 1
        if geht(arbeiten, k, x):
            return x, pruefungen

for k in [1, 2, 3, 4, 5, 20]:
    print(k, kleinstes_x_linear(ARBEITEN, k))
```

Das Ergebnis ist der kleinste Höchstwert. Aber die Zahl der Prüfungen ist groß: bei `k = 1` sind es 122 (so viel wie die Gesamtsumme), bei `k = 2` 67 und
bei vier Personen 34. Bei einem Aufwand von Millionen würde das viel zu lange dauern.

</details>

#### Aufgabe 3.3

Jetzt mit Halbieren. Schreibe `kleinstes_x_binaer(arbeiten, k)`. Wir wissen: `x` liegt zwischen `max(arbeiten)` (kleiner geht nie) und `sum(arbeiten)` (eine Person macht alles, das geht
immer). Halte zwei Grenzen `links = max(arbeiten)` und `rechts = sum(arbeiten)` fest. **Das Ergebnis liegt immer in `links` bis `rechts`.** Solange `links < rechts`: Prüfe `mitte = (links + rechts) // 2`.
Geht es mit `mitte`, dann ist `mitte` noch möglich, aber vielleicht gibt es ein kleineres: `rechts = mitte`. Sonst ist `mitte` zu klein: `links = mitte + 1`. Am Ende ist `links == rechts`.
Gib wieder `(x, pruefungen)` zurück.

**Erwartete Ausgabe:**
```
1 (122, 6)
2 (67, 6)
3 (44, 7)
4 (34, 7)
5 (26, 7)
20 (12, 7)
```

In [ ]:
ARBEITEN = [7, 2, 5, 10, 8, 3, 9, 4, 6, 1, 12, 5, 8, 2, 7, 9, 3, 11, 4, 6]     # Summe 122

def geht(arbeiten, k, x):
    personen = 1
    summe = 0
    for paket in arbeiten:
        if paket > x:
            return False
        if summe + paket > x:
            personen = personen + 1
            summe = paket
        else:
            summe = summe + paket
    return personen <= k

def kleinstes_x_binaer(arbeiten, k):
    links = max(arbeiten)
    rechts = sum(arbeiten)
    pruefungen = 0
    # dein Code
    return links, pruefungen

for k in [1, 2, 3, 4, 5, 20]:
    print(k, kleinstes_x_binaer(ARBEITEN, k))

<details>
<summary>Tipp anzeigen</summary>

`while links < rechts:` Bei `geht(...)` gilt `rechts = mitte` (**nicht** `mitte - 1`, `mitte` könnte die Antwort sein), sonst `links = mitte + 1`. Zähle `pruefungen` bei jedem Aufruf von `geht`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
ARBEITEN = [7, 2, 5, 10, 8, 3, 9, 4, 6, 1, 12, 5, 8, 2, 7, 9, 3, 11, 4, 6]     # Summe 122

def geht(arbeiten, k, x):
    personen = 1
    summe = 0
    for paket in arbeiten:
        if paket > x:
            return False
        if summe + paket > x:
            personen = personen + 1
            summe = paket
        else:
            summe = summe + paket
    return personen <= k

def kleinstes_x_binaer(arbeiten, k):
    links = max(arbeiten)            # kleiner geht nie
    rechts = sum(arbeiten)           # das geht immer
    pruefungen = 0
    while links < rechts:
        mitte = (links + rechts) // 2
        pruefungen = pruefungen + 1
        if geht(arbeiten, k, mitte):
            rechts = mitte           # mitte geht: kleineres suchen, mitte bleibt möglich
        else:
            links = mitte + 1        # mitte geht nicht: alles bis mitte scheidet aus
    return links, pruefungen

for k in [1, 2, 3, 4, 5, 20]:
    print(k, kleinstes_x_binaer(ARBEITEN, k))
```

Dieselben Ergebnisse wie bei der linearen Suche, aber nur noch 6 oder 7 Prüfungen statt 122, 67, 44, 34, 26 und 12. Bei Summen von Millionen wären es nur etwa 20 bis 30 Prüfungen.
Das Muster merkst du dir: Das **kleinste** `x`, für das etwas geht, findet man mit `if geht(mitte): rechts = mitte`, `else: links = mitte + 1`, Schleife `while links < rechts`.
Die Zahl der Prüfungen ist etwa log₂ der Breite des Bereichs, bei 122 − 12 = 110 ungefähr 7.

</details>

#### Aufgabe 3.4: Was wird ausgegeben?

Überlege zuerst, was der Code ausgibt, und schreibe es auf. Führe die Zelle erst danach aus.

In [ ]:
def ist_gut(x):
    return x == 4 or x >= 12              # nicht monoton: 4 geht, 5 bis 11 nicht, ab 12 wieder

def kleinstes_linear(links, rechts):
    for x in range(links, rechts + 1):
        if ist_gut(x):
            return x

def kleinstes_binaer(links, rechts):
    while links < rechts:
        mitte = (links + rechts) // 2
        if ist_gut(mitte):
            rechts = mitte
        else:
            links = mitte + 1
    return links

print(kleinstes_linear(1, 40), kleinstes_binaer(1, 40))

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

**Ausgabe:**
```
4 12
```

Ausgegeben wird `4 12`. Die lineare Suche findet das kleinste gute `x`, die 4. Die binäre Suche prüft nacheinander 20 (gut), 10 (nicht gut: `links = 11`), 15, 13, 12 (alle gut) und 11 (nicht gut):
Sie landet bei 12. Sie schließt aus „10 geht nicht“, dass **alles bis 10** nicht geht, aber die 4 geht ja. Die Prüfung ist nicht **monoton**: Aus „geht mit 4“ folgt nicht „geht mit 5“.
Die binäre Suche über den Zielwert darf man nur benutzen, wenn aus „geht mit `x`“ immer „geht mit jedem größeren `x`“ folgt (oder immer umgekehrt).

</details>

#### Frage 3.5

1. Warum ist die Prüfung `geht` in 3.1 monoton? Was bleibt an einer gültigen Verteilung richtig, wenn man `x` vergrößert?
2. Warum darf man das gierige Verfahren in `geht` nehmen? Warum ist es nicht schlimm, dass die erste Person so viel wie möglich bekommt? Tipp: Austauschargument aus Notebook 12.
3. Für welche der folgenden Aufgaben darf man den Zielwert raten und halbieren? (a) die kleinste Zahl `x`, so dass `x * x >= 5000`; (b) die kleinste Zahl `x`, die ein Teiler von 5000 und `x > 1`;
   (c) die höchste Mindestbewertung für Hotels (man darf nur Hotels mit Bewertung mindestens `x` nehmen).

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Hat man mit der Grenze `x` eine Verteilung gefunden, dann gilt sie auch für jedes größere `x`, denn jede Person hat dann höchstens so viel, wie vorher. Die gleiche Verteilung bleibt gültig.
2. Austauschargument: Gibt es irgendeine gültige Verteilung, dann kann man die Grenze zwischen der ersten und zweiten Person so weit nach rechts schieben, wie es für die erste Person noch passt. Die zweite Person
   bekommt dann weniger Pakete, das schadet ihr nicht. Die Pakete der folgenden Personen verschieben sich nur nach hinten und keiner bekommt mehr als vorher. Also lohnt es sich nie, einer Person weniger zu geben als möglich.
3. (a) Ja: für größere `x` stimmt `x * x >= 5000` immer weiter, die Prüfung ist monoton (das Ergebnis ist 71). (b) Nein: ob eine Zahl ein Teiler ist, springt hin und her, aus „2 ist Teiler“ folgt nichts über 3.
   (c) Ja, aber in der **anderen Richtung**: Geht es mit Mindestbewertung `x`, dann auch mit jedem kleineren `x`, denn dann sind mehr Hotels erlaubt. Gesucht ist hier das **größte** `x`, das noch geht.
   Die Schleife sieht gespiegelt aus (siehe Wettbewerbs-Ecke).

</details>

## 4. Meet in the Middle: Gewichte kombinieren

Eine Balkenwaage hat Gewichtsstücke. Jedes Stück darf **links** liegen (bei der Ware), **rechts** liegen oder **ganz weg** bleiben. Liegt die Ware links und sind die Stücke
im Gleichgewicht, dann gilt: *Ware + (Stücke links) = (Stücke rechts)*, also *Ware = (Stücke rechts) − (Stücke links)*. Jedes Stück hat also den Faktor **+1** (rechts), **−1** (links) oder **0** (weg),
und die Ware wiegt die Summe aus Gewicht mal Faktor.

Mit den Stücken 1, 3 und 9 gibt es 3 · 3 · 3 = 27 Verteilungen. Bei `n` Stücken sind es **3ⁿ**. Das wächst sehr schnell: bei 12 Stücken 531 441, bei 20 Stücken über 3 Milliarden.

#### Aufgabe 4.1

Schreibe `alle_summen(stuecke)`: Sie gibt die Liste der Summen **aller** Verteilungen zurück (je Stück Faktor −1, 0 oder 1). Fange mit der Liste `[0]` an. Für jedes Gewicht entsteht
eine neue Liste: Jede bisherige Summe `s` wird dreimal übernommen, als `s - gewicht`, `s` und `s + gewicht`.

**Erwartete Ausgabe:**
```
27
[-13, -12, -11, -10, -9, -8, -7, -6, -5, -4, -3, -2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
```

In [ ]:
def alle_summen(stuecke):
    summen = [0]
    # dein Code
    return summen

s = alle_summen([1, 3, 9])
print(len(s))
print(sorted(set(s)))

<details>
<summary>Tipp anzeigen</summary>

`for gewicht in stuecke:` Eine neue leere Liste `neu`. Für jedes `s in summen` und jeden `faktor in (-1, 0, 1)` hänge `s + faktor * gewicht` an. Dann `summen = neu`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def alle_summen(stuecke):
    summen = [0]
    for gewicht in stuecke:
        neu = []
        for s in summen:
            for faktor in (-1, 0, 1):
                neu.append(s + faktor * gewicht)
        summen = neu
    return summen

s = alle_summen([1, 3, 9])
print(len(s))
print(sorted(set(s)))
```

Es sind 27 Summen. Die verschiedenen Werte sind alle ganzen Zahlen von −13 bis 13: Mit den Stücken 1, 3 und 9 kann man **jedes** Gewicht von 1 bis 13 wiegen. (Das nennt man das ausgeglichene
Dreiersystem: jede Zahl lässt sich mit den Ziffern −1, 0, 1 zur Basis 3 schreiben.)

</details>

#### Aufgabe 4.2

Die Summen aller Verteilungen sind berechnet. Schreibe `wiegbar_alle(summen, ziel)`: `True`, wenn eine Ware mit dem Gewicht `ziel` gewogen werden kann, also wenn `ziel`
unter den Summen vorkommt. Für viele Fragen mit derselben Liste lohnt sich eine **Menge** (`set`), in der das Nachschlagen schnell geht; `set(summen)` macht aus der Liste eine Menge, in der `in` schnell geht. Baue sie bei jedem Aufruf neu, das
reicht hier zum Verstehen.

**Erwartete Ausgabe:**
```
531441
True True False
```

In [ ]:
def alle_summen(stuecke):
    summen = [0]
    for gewicht in stuecke:
        neu = []
        for s in summen:
            for faktor in (-1, 0, 1):
                neu.append(s + faktor * gewicht)
        summen = neu
    return summen

STUECKE = [2, 3, 7, 11, 13, 20, 29, 41, 53, 67, 80, 97]       # 12 Stücke: 3**12 = 531441 Verteilungen

def wiegbar_alle(summen, ziel):
    # dein Code
    return False

summen = alle_summen(STUECKE)
print(len(summen))
print(wiegbar_alle(summen, 100), wiegbar_alle(summen, 300), wiegbar_alle(summen, 450))

<details>
<summary>Tipp anzeigen</summary>

`return ziel in set(summen)`

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def alle_summen(stuecke):
    summen = [0]
    for gewicht in stuecke:
        neu = []
        for s in summen:
            for faktor in (-1, 0, 1):
                neu.append(s + faktor * gewicht)
        summen = neu
    return summen

STUECKE = [2, 3, 7, 11, 13, 20, 29, 41, 53, 67, 80, 97]       # 12 Stücke: 3**12 = 531441 Verteilungen

def wiegbar_alle(summen, ziel):
    return ziel in set(summen)

summen = alle_summen(STUECKE)
print(len(summen))
print(wiegbar_alle(summen, 100), wiegbar_alle(summen, 300), wiegbar_alle(summen, 450))
```

Die Liste hat 531 441 Einträge, das Aufbauen dauert spürbar und braucht viel Speicher. Die Frage selbst ist dann schnell beantwortet. Die Gewichte 100 und 300 lassen sich wiegen, die 450 nicht: Alle Stücke zusammen wiegen nur 423 (`2 + 3 + … + 97`), mehr geht nie.

</details>

#### Aufgabe 4.3

Jetzt **Meet in the Middle**. Teile die Stücke in zwei Hälften `vorn = stuecke[:h]` und `hinten = stuecke[h:]` mit `h = len(stuecke) // 2`. Berechne mit `alle_summen` die Summen jeder Hälfte (bei 12 Stücken
je 3⁶ = 729). Jede Verteilung aller Stücke besteht aus einer Verteilung der vorderen **und** einer der hinteren Hälfte, und ihre Summe ist `a + b`. Es gilt also `ziel = a + b` genau dann,
wenn `ziel - a` unter den Summen der hinteren Hälfte vorkommt. Lege die hinteren Summen in eine Menge `menge_hinten` und gehe alle `a` der vorderen Hälfte durch.

Schreibe `wiegbar_mitte(stuecke, ziel)`. Sie gibt ein Paar `(ja, anzahl)` zurück, `ja` ist `True` oder `False`, `anzahl` ist die Zahl der berechneten Summen (beide Hälften zusammen).

**Erwartete Ausgabe:**
```
100 (True, 1458)
300 (True, 1458)
450 (False, 1458)
500 von 500 Antworten stimmen mit der Menge aller Summen überein
```

In [ ]:
def alle_summen(stuecke):
    summen = [0]
    for gewicht in stuecke:
        neu = []
        for s in summen:
            for faktor in (-1, 0, 1):
                neu.append(s + faktor * gewicht)
        summen = neu
    return summen

STUECKE = [2, 3, 7, 11, 13, 20, 29, 41, 53, 67, 80, 97]       # 12 Stücke: 3**12 = 531441 Verteilungen

def wiegbar_mitte(stuecke, ziel):
    h = len(stuecke) // 2
    # dein Code
    return False, 0

for ziel in [100, 300, 450]:
    print(ziel, wiegbar_mitte(STUECKE, ziel))
menge = set(alle_summen(STUECKE))
gleich = 0
for ziel in range(1, 501):
    if wiegbar_mitte(STUECKE, ziel)[0] == (ziel in menge):
        gleich = gleich + 1
print(gleich, 'von 500 Antworten stimmen mit der Menge aller Summen überein')

<details>
<summary>Tipp anzeigen</summary>

`summen_vorn = alle_summen(stuecke[:h])`, `summen_hinten = alle_summen(stuecke[h:])`, `menge_hinten = set(summen_hinten)`. Dann `for a in summen_vorn:` und `if ziel - a in menge_hinten:`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def alle_summen(stuecke):
    summen = [0]
    for gewicht in stuecke:
        neu = []
        for s in summen:
            for faktor in (-1, 0, 1):
                neu.append(s + faktor * gewicht)
        summen = neu
    return summen

STUECKE = [2, 3, 7, 11, 13, 20, 29, 41, 53, 67, 80, 97]       # 12 Stücke: 3**12 = 531441 Verteilungen

def wiegbar_mitte(stuecke, ziel):
    h = len(stuecke) // 2
    summen_vorn = alle_summen(stuecke[:h])           # 3**h Summen
    summen_hinten = alle_summen(stuecke[h:])         # 3**(n-h) Summen
    menge_hinten = set(summen_hinten)
    anzahl = len(summen_vorn) + len(summen_hinten)
    for a in summen_vorn:
        if ziel - a in menge_hinten:                 # a + b = ziel
            return True, anzahl
    return False, anzahl

for ziel in [100, 300, 450]:
    print(ziel, wiegbar_mitte(STUECKE, ziel))
menge = set(alle_summen(STUECKE))
gleich = 0
for ziel in range(1, 501):
    if wiegbar_mitte(STUECKE, ziel)[0] == (ziel in menge):
        gleich = gleich + 1
print(gleich, 'von 500 Antworten stimmen mit der Menge aller Summen überein')
```

Statt 531 441 Summen werden nur 729 + 729 = **1 458** berechnet, rund 365-mal weniger. Das Ergebnis ist dasselbe: Alle 500 Antworten stimmen mit der großen Menge überein. Die Rechnung
ist `3^(n/2) + 3^(n/2)` statt `3^n`: Wurzel der Anzahl der Möglichkeiten.

</details>

#### Aufgabe 4.4

Was tut man, wenn `ziel` **nicht** genau zu wiegen ist und man wissen will, wie nahe man herankommt? Für jedes `a` der vorderen Hälfte sucht man in den **sortierten** hinteren
Summen den Wert `b`, der `ziel - a` am nächsten liegt. Das geht mit `bisect_left`: Die Stelle `i = bisect_left(hinten_sortiert, ziel - a)` hat zwei Nachbarn,
`hinten_sortiert[i - 1]` (kleiner) und `hinten_sortiert[i]` (größer oder gleich). Der nähere der beiden ist der beste Partner (aufpassen: `i` kann 0 oder `len(...)` sein).

Schreibe `naechste_summe(stuecke, ziel)`: Sie gibt die Summe `a + b` zurück, die am nächsten an `ziel` liegt (bei Gleichstand die kleinere). Die letzte Spalte der Ausgabe vergleicht mit dem
Durchprobieren über alle Summen: Sie muss immer `True` sein.

**Erwartete Ausgabe:**
```
100 100 True
450 423 True
500 423 True
```

In [ ]:
def alle_summen(stuecke):
    summen = [0]
    for gewicht in stuecke:
        neu = []
        for s in summen:
            for faktor in (-1, 0, 1):
                neu.append(s + faktor * gewicht)
        summen = neu
    return summen

STUECKE = [2, 3, 7, 11, 13, 20, 29, 41, 53, 67, 80, 97]       # 12 Stücke: 3**12 = 531441 Verteilungen

from bisect import bisect_left

def naechste_summe(stuecke, ziel):
    h = len(stuecke) // 2
    summen_vorn = alle_summen(stuecke[:h])
    hinten_sortiert = sorted(alle_summen(stuecke[h:]))
    beste = None                       # Paar (Abweichung, Summe)
    # dein Code
    return 0

menge = set(alle_summen(STUECKE))
for ziel in [100, 450, 500]:
    gefunden = naechste_summe(STUECKE, ziel)
    beste = min(abs(s - ziel) for s in menge)
    print(ziel, gefunden, abs(gefunden - ziel) == beste)

<details>
<summary>Tipp anzeigen</summary>

Für jedes `a`: `i = bisect_left(hinten_sortiert, ziel - a)`, dann `for j in (i - 1, i):` und `if 0 <= j < len(hinten_sortiert):` die Summe `a + hinten_sortiert[j]` bilden. Vergleiche Paare
`(abs(summe - ziel), summe)`: Das kleinere Paar hat die kleinere Abweichung, bei Gleichstand die kleinere Summe.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def alle_summen(stuecke):
    summen = [0]
    for gewicht in stuecke:
        neu = []
        for s in summen:
            for faktor in (-1, 0, 1):
                neu.append(s + faktor * gewicht)
        summen = neu
    return summen

STUECKE = [2, 3, 7, 11, 13, 20, 29, 41, 53, 67, 80, 97]       # 12 Stücke: 3**12 = 531441 Verteilungen

from bisect import bisect_left

def naechste_summe(stuecke, ziel):
    h = len(stuecke) // 2
    summen_vorn = alle_summen(stuecke[:h])
    hinten_sortiert = sorted(alle_summen(stuecke[h:]))
    beste = None                       # Paar (Abweichung, Summe)
    for a in summen_vorn:
        i = bisect_left(hinten_sortiert, ziel - a)
        for j in (i - 1, i):           # die beiden Nachbarn
            if 0 <= j < len(hinten_sortiert):
                summe = a + hinten_sortiert[j]
                kandidat = (abs(summe - ziel), summe)
                if beste is None or kandidat < beste:
                    beste = kandidat
    return beste[1]

menge = set(alle_summen(STUECKE))
for ziel in [100, 450, 500]:
    gefunden = naechste_summe(STUECKE, ziel)
    beste = min(abs(s - ziel) for s in menge)
    print(ziel, gefunden, abs(gefunden - ziel) == beste)
```

Für 100 gibt es einen Treffer, für 450 und 500 ist 423 die nächste Summe (Abweichung 27 und 77): Mehr als alle Stücke zusammen kann man nicht wiegen. Die letzte Spalte ist dreimal `True`, das Ergebnis hat dieselbe Abweichung wie das Durchprobieren über alle 531 441 Summen. Die hinteren Summen zu sortieren kostet einmal O(m log m) mit m = 729, jede Suche danach nur O(log m).

</details>

#### Frage 4.5

1. Wie viele Verteilungen gibt es bei 30 Stücken, und wie viele Summen müsste Meet in the Middle berechnen? Und bei 40 Stücken? Was heißt das für die Grenze des Verfahrens?
2. Warum sortiert man die hinteren Summen (oder legt sie in eine Menge), statt für jedes `a` alle `b` durchzugehen?
3. Meet in the Middle hilft auch bei Teilmengen: „Gibt es unter 40 Zahlen eine Teilmenge mit Summe 1 000 000?“ Wie viele Teilmengen hat man, und wie viele Summen bei der geteilten Rechnung?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Bei 30 Stücken sind es 3³⁰ = 205 891 132 094 649 (rund 2 · 10¹⁴) Verteilungen, aber nur 2 · 3¹⁵ = 28 697 814 Summen: machbar, wenn auch langsam und speicherhungrig. Bei 40 Stücken sind es
   12 157 665 459 056 928 801 (mehr als 10¹⁹) Verteilungen gegen 2 · 3²⁰ = 6 973 568 802 Summen: In Python nicht mehr zu schaffen. Meet in the Middle **verschiebt** die Grenze von etwa
   n = 14 auf etwa n = 28 bis 30, löst aber nicht beliebig große Aufgaben.
2. Geht man für jedes `a` alle `b` durch, sind es 3^(n/2) · 3^(n/2) = 3ⁿ Paare: Man hat nichts gewonnen. Mit Menge (Nachschlagen in O(1)) oder sortierter Liste mit `bisect` (O(log m)) pro `a` bleibt es
   bei etwa 3^(n/2) Schritten (die Sortierung kostet O(m log m) mit m = 3^(n/2)).
3. 2⁴⁰ ≈ 1,1 Billionen Teilmengen, geteilt dagegen 2 · 2²⁰ = 2 097 152 Summen. Das ist leicht machbar: Berechne alle Summen der ersten und der zweiten Hälfte und suche zu jeder Summe `a` der ersten
   Hälfte den Partner `1 000 000 - a` in der zweiten.

</details>

## Wann benutzt man binäre Suche?

1. **Eine sortierte Liste und viele Fragen**: `bisect_left` und `bisect_right` (O(log n)), oder eine Menge/ein Wörterbuch, wenn es nur um „kommt vor?“ geht.
2. **Ein Zielwert und eine monotone Prüfung** („Geht es mit höchstens `x`?“): Die Grenze durch Halbieren suchen. Vorher klären: Steht in `links` und `rechts` eine Lösung beziehungsweise eine
   Nicht-Lösung? Wird der Bereich in jeder Runde **kleiner**? Ist die Prüfung wirklich monoton? Ist der gierige Test richtig (Notebook 12)?
3. **Zu viele Möglichkeiten, aber in zwei Hälften zerlegbar**: Meet in the Middle. Statt `k^n` nur `2 · k^(n/2)` Summen, dazu Nachschlagen (Menge) oder nächster Nachbar (`bisect`).

Grenzen: Die Liste muss sortiert sein, die Prüfung monoton, und bei Meet in the Middle müssen die Hälften unabhängig sein (die Summe zerfällt in die Summe der Hälften).

## Wettbewerbs-Ecke 1: Vollgeladen

*Angelehnt an Aufgabe 2 „Vollgeladen“ der 1. Runde des 40. Bundeswettbewerbs Informatik (2021/22).*

Familie Meier fährt in Etappen zu Verwandten: Die Reise dauert `gesamt` Minuten reine Fahrzeit. Entlang der Route liegen Hotels, zu jedem gehören die **Position** (Fahrminuten vom Start) und die **Bewertung**.
Pro Tag darf man **höchstens 360 Minuten** fahren (sechs Stunden), und die Reise darf **höchstens fünf Tage** dauern, es gibt also **höchstens 4 Übernachtungen**. Übernachtet wird nur in Hotels.
Gesucht ist die Auswahl von Hotels, bei der die **niedrigste Bewertung** unter den gewählten Hotels **so hoch wie möglich** ist.

Die Dateien `daten/hotels1.txt` bis `hotels5.txt` sind die Beispiele der Aufgabe. Erste Zeile: Zahl der Hotels, zweite Zeile: `gesamt`, dann je Hotel eine Zeile `position bewertung`.

Wir lösen die Aufgabe in zwei Schritten: **Zielwert raten** („Geht die Reise, wenn ich nur Hotels mit Bewertung mindestens `x` benutze?“), das prüft ein gieriges Verfahren, und dann die **binäre Suche**
über `x`.

#### Aufgabe E.1

Schreibe `lies_hotels(dateiname)`: Lies die Datei ein und gib `(gesamt, hotels)` zurück. `hotels` ist eine nach der **Position** sortierte Liste von
Paaren `(position, bewertung)` (`int` und `float`). Der Aufruf unten zeigt, was herauskommen soll.

**Erwartete Ausgabe:**
```
12 1680 [(12, 4.3), (326, 4.8), (347, 2.7)]
```

In [ ]:
MAX_MINUTEN = 360                 # sechs Stunden pro Tag
NAECHTE = 4                       # höchstens fünf Tage, also vier Übernachtungen

def lies_hotels(dateiname):
    gesamt = 0
    hotels = []
    # dein Code
    return gesamt, hotels

gesamt, hotels = lies_hotels('daten/hotels1.txt')
print(len(hotels), gesamt, hotels[:3])

<details>
<summary>Tipp anzeigen</summary>

`zahlen = datei.read().split()` liefert alle Wörter der Datei als Liste. `zahlen[0]` ist die Anzahl, `zahlen[1]` die Gesamtfahrzeit, ab `zahlen[2]` stehen Position und Bewertung abwechselnd. Mit
`hotels.sort()` sortierst du Paare nach dem ersten Element.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
MAX_MINUTEN = 360                 # sechs Stunden pro Tag
NAECHTE = 4                       # höchstens fünf Tage, also vier Übernachtungen

def lies_hotels(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        zahlen = datei.read().split()
    gesamt = int(zahlen[1])
    hotels = []
    for i in range(int(zahlen[0])):
        hotels.append((int(zahlen[2 + 2 * i]), float(zahlen[3 + 2 * i])))
    hotels.sort()                     # nach der Position
    return gesamt, hotels

gesamt, hotels = lies_hotels('daten/hotels1.txt')
print(len(hotels), gesamt, hotels[:3])
```

Es sind 12 Hotels und 1 680 Minuten Gesamtfahrzeit.

</details>

#### Aufgabe E.2

Schreibe `reise_moeglich(hotels, gesamt, x)`: Sie gibt die Liste der gewählten Hotels zurück oder `None`, wenn die Reise mit Hotels der Bewertung **mindestens `x`** nicht möglich ist. Gieriges Verfahren:
Du stehst bei `pos = 0`. Solange das Ziel mehr als 360 Minuten entfernt ist (`gesamt - pos > MAX_MINUTEN`), fahre bis zum **am weitesten entfernten** guten Hotel mit `pos < position <= pos + MAX_MINUTEN`
und übernachte dort (höchstens `NAECHTE`-mal; gibt es keins in Reichweite, geht es nicht).

**Erwartete Ausgabe:**
```
0.0 [(359, 2.6), (687, 4.4), (1008, 2.6), (1360, 2.8)]
2.7 [(347, 2.7), (687, 4.4), (1007, 2.8), (1360, 2.8)]
2.8 None
```

In [ ]:
MAX_MINUTEN = 360                 # sechs Stunden pro Tag
NAECHTE = 4                       # höchstens fünf Tage, also vier Übernachtungen

def lies_hotels(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        zahlen = datei.read().split()
    gesamt = int(zahlen[1])
    hotels = []
    for i in range(int(zahlen[0])):
        hotels.append((int(zahlen[2 + 2 * i]), float(zahlen[3 + 2 * i])))
    hotels.sort()                     # nach der Position
    return gesamt, hotels

def reise_moeglich(hotels, gesamt, x):
    gut = [h for h in hotels if h[1] >= x]
    pos = 0
    gewaehlt = []
    # dein Code
    return None

gesamt, hotels = lies_hotels('daten/hotels1.txt')
for x in [0.0, 2.7, 2.8]:
    print(x, reise_moeglich(hotels, gesamt, x))

<details>
<summary>Tipp anzeigen</summary>

`gut` enthält nur Hotels mit `bewertung >= x`. `for nacht in range(NAECHTE):` Ist `gesamt - pos <= MAX_MINUTEN`, gib `gewaehlt` zurück. Suche sonst in `gut` das **letzte** Hotel (es ist sortiert) mit
`pos < hotel[0] <= pos + MAX_MINUTEN`; gibt es keins, `return None`. Hänge es an und setze `pos`. Nach der Schleife prüfe noch einmal `gesamt - pos <= MAX_MINUTEN`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
MAX_MINUTEN = 360                 # sechs Stunden pro Tag
NAECHTE = 4                       # höchstens fünf Tage, also vier Übernachtungen

def lies_hotels(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        zahlen = datei.read().split()
    gesamt = int(zahlen[1])
    hotels = []
    for i in range(int(zahlen[0])):
        hotels.append((int(zahlen[2 + 2 * i]), float(zahlen[3 + 2 * i])))
    hotels.sort()                     # nach der Position
    return gesamt, hotels

def reise_moeglich(hotels, gesamt, x):
    gut = [h for h in hotels if h[1] >= x]
    pos = 0
    gewaehlt = []
    for nacht in range(NAECHTE):
        if gesamt - pos <= MAX_MINUTEN:
            return gewaehlt                         # das Ziel ist erreichbar
        weitestes = None
        for hotel in gut:
            if pos < hotel[0] <= pos + MAX_MINUTEN:
                weitestes = hotel                   # gut ist sortiert: das letzte passende ist das weiteste
        if weitestes is None:
            return None                             # kein Hotel in Reichweite
        gewaehlt.append(weitestes)
        pos = weitestes[0]
    if gesamt - pos <= MAX_MINUTEN:
        return gewaehlt
    return None

gesamt, hotels = lies_hotels('daten/hotels1.txt')
for x in [0.0, 2.7, 2.8]:
    print(x, reise_moeglich(hotels, gesamt, x))
```

Ohne Einschränkung (`x = 0.0`) fährt das Verfahren immer bis zum weitesten Hotel, auch wenn es schlecht bewertet ist (die niedrigste Bewertung ist hier 2.6). Mit `x = 2.7` bleiben nur Hotels ab
2.7, und es geht noch: bei 347, 687, 1007 und 1360. Mit `x = 2.8` geht es nicht mehr.

</details>

#### Frage E.3

1. Warum darf man in `reise_moeglich` immer das **am weitesten entfernte** Hotel nehmen? Was hat man davon, am nächsten Tag weiter vorn zu stehen? (Austauschargument)
2. Warum ist `reise_moeglich(..., x)` **monoton**: Aus welchem Ergebnis für `x` folgt etwas für größere oder für kleinere `x`?
3. Welche Werte von `x` muss man überhaupt ausprobieren? Hängt das Ergebnis der Suche von Zahlen zwischen zwei vorkommenden Bewertungen ab?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Angenommen, es gibt irgendeine gültige Reise, die am ersten Tag **vor** dem weitesten guten Hotel übernachtet. Tauscht man dieses Hotel gegen das weiteste, steht die Familie
   mindestens so weit vorn, und die Strecke bis zum nächsten Hotel kann nur **kürzer** werden. Die Reise bleibt gültig. Wer weiter vorn steht, hat es später nicht schwerer.
2. Geht es mit der Mindestbewertung `x`, dann geht es mit jedem **kleineren** `x` auch, denn dann sind mehr Hotels erlaubt und die gleiche Auswahl bleibt gültig. Gesucht ist das **größte** `x`, für das es noch geht
   (gespiegelt zu 3.3, wo das kleinste gesucht war).
3. Nur die Bewertungen, die in der Datei vorkommen: Zwischen zwei vorkommenden Bewertungen ändert sich die Menge der guten Hotels nicht. Man sortiert also die **verschiedenen** Bewertungen und sucht in dieser Liste.

</details>

#### Aufgabe E.4

Jetzt die binäre Suche über die Bewertungen. `bewertungen = sorted(set(h[1] for h in hotels))` ist die aufsteigende Liste der **verschiedenen** Bewertungen. Das Ergebnis ist die **größte** Bewertung `x`
in dieser Liste, mit der `reise_moeglich` noch klappt. Mit der kleinsten Bewertung (alle Hotels erlaubt) klappt es in den Beispielen immer.

Halte zwei Grenzen fest: `links` ist ein Index, **für den es sicher geht** (am Anfang 0), `rechts` ein Index, ab dem es **nicht** mehr geht (am Anfang `len(bewertungen)`). Solange `links + 1 < rechts`: Prüfe
`mitte = (links + rechts) // 2`. Geht es, ist `links = mitte`, sonst `rechts = mitte`. Am Ende ist `bewertungen[links]` das Ergebnis.

Schreibe `beste_bewertung(hotels, gesamt)`. Sie gibt `(x, gewaehlt, pruefungen)` zurück (die Absicherung am Anfang zählt als eine Prüfung, schon eingetragen). `beste_linear` zählt zum Vergleich die Prüfungen,
wenn man die Bewertungen von oben nach unten durchprobiert.

**Erwartete Ausgabe:**
```
hotels1.txt: 12 Hotels, niedrigste Bewertung 2.7 , Prüfungen: binär 5 , linear 7
    [(347, 2.7), (687, 4.4), (1007, 2.8), (1360, 2.8)]
hotels2.txt: 25 Hotels, niedrigste Bewertung 2.3 , Prüfungen: binär 6 , linear 10
    [(341, 2.3), (700, 3.0), (1053, 4.8), (1380, 5.0)]
hotels3.txt: 500 Hotels, niedrigste Bewertung 0.3 , Prüfungen: binär 7 , linear 48
    [(360, 1.3), (717, 0.3), (1076, 3.8), (1433, 1.7)]
hotels4.txt: 500 Hotels, niedrigste Bewertung 4.6 , Prüfungen: binär 7 , linear 5
    [(340, 4.6), (676, 4.6), (1032, 4.9), (1316, 4.9)]
hotels5.txt: 1500 Hotels, niedrigste Bewertung 5.0 , Prüfungen: binär 7 , linear 1
    [(317, 5.0), (636, 5.0), (987, 5.0), (1286, 5.0)]
```

In [ ]:
MAX_MINUTEN = 360                 # sechs Stunden pro Tag
NAECHTE = 4                       # höchstens fünf Tage, also vier Übernachtungen

def lies_hotels(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        zahlen = datei.read().split()
    gesamt = int(zahlen[1])
    hotels = []
    for i in range(int(zahlen[0])):
        hotels.append((int(zahlen[2 + 2 * i]), float(zahlen[3 + 2 * i])))
    hotels.sort()                     # nach der Position
    return gesamt, hotels

def reise_moeglich(hotels, gesamt, x):
    gut = [h for h in hotels if h[1] >= x]
    pos = 0
    gewaehlt = []
    for nacht in range(NAECHTE):
        if gesamt - pos <= MAX_MINUTEN:
            return gewaehlt                         # das Ziel ist erreichbar
        weitestes = None
        for hotel in gut:
            if pos < hotel[0] <= pos + MAX_MINUTEN:
                weitestes = hotel                   # gut ist sortiert: das letzte passende ist das weiteste
        if weitestes is None:
            return None                             # kein Hotel in Reichweite
        gewaehlt.append(weitestes)
        pos = weitestes[0]
    if gesamt - pos <= MAX_MINUTEN:
        return gewaehlt
    return None

def beste_linear(hotels, gesamt):
    """Probiert die Bewertungen von der besten zur schlechtesten und zählt die Prüfungen."""
    bewertungen = sorted(set(h[1] for h in hotels), reverse=True)
    pruefungen = 0
    for x in bewertungen:
        pruefungen = pruefungen + 1
        if reise_moeglich(hotels, gesamt, x) is not None:
            return pruefungen
    return pruefungen

def beste_bewertung(hotels, gesamt):
    bewertungen = sorted(set(h[1] for h in hotels))
    pruefungen = 1                     # die Absicherung: mit allen Hotels muss es gehen
    links = 0
    rechts = len(bewertungen)
    # dein Code
    x = bewertungen[links]
    return x, reise_moeglich(hotels, gesamt, x), pruefungen

for nr in range(1, 6):
    gesamt, hotels = lies_hotels('daten/hotels' + str(nr) + '.txt')
    x, gewaehlt, binaer = beste_bewertung(hotels, gesamt)
    linear = beste_linear(hotels, gesamt)
    print('hotels' + str(nr) + '.txt:', len(hotels), 'Hotels, niedrigste Bewertung', x, ', Prüfungen: binär', binaer, ', linear', linear)
    print('   ', gewaehlt)

<details>
<summary>Tipp anzeigen</summary>

`while links + 1 < rechts:` Setze `mitte = (links + rechts) // 2`, zähle `pruefungen` hoch und prüfe `reise_moeglich(hotels, gesamt, bewertungen[mitte]) is not None`. Bei `True` gilt `links = mitte`, sonst
`rechts = mitte`. Hier ist **links die sichere Seite**, deshalb `mitte` und nicht `mitte + 1`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
MAX_MINUTEN = 360                 # sechs Stunden pro Tag
NAECHTE = 4                       # höchstens fünf Tage, also vier Übernachtungen

def lies_hotels(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        zahlen = datei.read().split()
    gesamt = int(zahlen[1])
    hotels = []
    for i in range(int(zahlen[0])):
        hotels.append((int(zahlen[2 + 2 * i]), float(zahlen[3 + 2 * i])))
    hotels.sort()                     # nach der Position
    return gesamt, hotels

def reise_moeglich(hotels, gesamt, x):
    gut = [h for h in hotels if h[1] >= x]
    pos = 0
    gewaehlt = []
    for nacht in range(NAECHTE):
        if gesamt - pos <= MAX_MINUTEN:
            return gewaehlt                         # das Ziel ist erreichbar
        weitestes = None
        for hotel in gut:
            if pos < hotel[0] <= pos + MAX_MINUTEN:
                weitestes = hotel                   # gut ist sortiert: das letzte passende ist das weiteste
        if weitestes is None:
            return None                             # kein Hotel in Reichweite
        gewaehlt.append(weitestes)
        pos = weitestes[0]
    if gesamt - pos <= MAX_MINUTEN:
        return gewaehlt
    return None

def beste_linear(hotels, gesamt):
    """Probiert die Bewertungen von der besten zur schlechtesten und zählt die Prüfungen."""
    bewertungen = sorted(set(h[1] for h in hotels), reverse=True)
    pruefungen = 0
    for x in bewertungen:
        pruefungen = pruefungen + 1
        if reise_moeglich(hotels, gesamt, x) is not None:
            return pruefungen
    return pruefungen

def beste_bewertung(hotels, gesamt):
    bewertungen = sorted(set(h[1] for h in hotels))
    pruefungen = 1                     # die Absicherung: mit allen Hotels muss es gehen
    links = 0                          # hier geht es sicher
    rechts = len(bewertungen)          # ab hier geht es nicht
    while links + 1 < rechts:
        mitte = (links + rechts) // 2
        pruefungen = pruefungen + 1
        if reise_moeglich(hotels, gesamt, bewertungen[mitte]) is not None:
            links = mitte
        else:
            rechts = mitte
    x = bewertungen[links]
    return x, reise_moeglich(hotels, gesamt, x), pruefungen

for nr in range(1, 6):
    gesamt, hotels = lies_hotels('daten/hotels' + str(nr) + '.txt')
    x, gewaehlt, binaer = beste_bewertung(hotels, gesamt)
    linear = beste_linear(hotels, gesamt)
    print('hotels' + str(nr) + '.txt:', len(hotels), 'Hotels, niedrigste Bewertung', x, ', Prüfungen: binär', binaer, ', linear', linear)
    print('   ', gewaehlt)
```

Die Ergebnisse (niedrigste Bewertung 2.7, 2.3, 0.3, 4.6 und 5.0) stimmen mit dem Lösungsnotebook zu Vollgeladen überein. Bei den Dateien 1 bis 3 spart die binäre Suche (5, 6 und 7 Prüfungen gegen 7, 10 und 48). Bei Datei 4 und 5 liegt
die Antwort ganz oben, dort ist die lineare Suche von oben schneller (5 und 1 Prüfung). Die binäre Suche braucht aber **immer** höchstens 7 Prüfungen: 50 oder 51 verschiedene Bewertungen, 2⁶ = 64.

</details>

#### Frage E.5

1. Die binäre Suche braucht bei allen fünf Dateien höchstens 7 Prüfungen, die lineare Suche (von oben nach unten) bei Datei 3 dagegen 48. Warum sind es bei Datei 5 nur 1 Prüfung für die lineare Suche?
   Welches Verfahren würdest du abgeben, wenn du die Daten nicht kennst?
2. Wie viele Prüfungen braucht die binäre Suche höchstens, wenn es 1 000 verschiedene Bewertungen gibt? Und was ändert sich, wenn die Reise 6 statt 5 Tage dauern darf (`NAECHTE = 5`)?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Bei Datei 5 klappt die Reise schon mit der besten Bewertung 5.0, die allererste Prüfung ist also erfolgreich. Bei Datei 3 liegt die Antwort ganz unten (0.3). Die lineare Suche hängt vom **Glück der Daten** ab, die binäre
   Suche braucht **immer** höchstens etwa log₂ der Zahl der Bewertungen. Also lieber die binäre Suche, vor allem bei großen Dateien.
2. Bei 1 000 Bewertungen etwa 10 Prüfungen (2¹⁰ = 1 024) plus die Absicherung. Mit `NAECHTE = 5` verändert sich das Verfahren nicht, nur die Konstante. In den echten Beispielen kann das Ergebnis dadurch aber **viel besser**
   werden (bei Datei 3 von 0.3 auf 4.8, siehe das Lösungsnotebook): Die Anzahl der Übernachtungen genau nach der Aufgabe wählen (höchstens fünf Tage sind vier Nächte).

</details>

## Wettbewerbs-Ecke 2: Marktwaage (für Schnelle)

*Angelehnt an Aufgabe 5 „Marktwaage“ der 1. Runde des 40. Bundeswettbewerbs Informatik (2021/22).*

Eine Marktwaage hat Gewichtsstücke. Die Ware liegt **links**, jedes Stück darf links (zur Ware), rechts oder gar nicht auf die Waage. Welche Gewichte kann man damit genau wiegen? In der Aufgabe sind das alle Vielfachen von
10 g von 10 g bis 10 kg. Wir arbeiten wie in Abschnitt 4 mit **Faktoren** −1, 0, +1 je Stück.

Die Datei `daten/waage_stuecke.txt` hat in der ersten Zeile die Zahl der Stücke, dann je Zeile ein Stück in Gramm. Es sind 12 Stücke (jedes einzeln, auch gleiche Gewichte stehen mehrfach in der Datei). Das gibt
3¹² = 531 441 Verteilungen. Mit Meet in the Middle brauchst du nur zweimal 729 Summen.

#### Aufgabe F.1

Schreibe `lies_stuecke(dateiname)`: Sie gibt die Liste der Gewichte zurück (die erste Zahl der Datei ist die Anzahl und gehört nicht zur Liste).

**Erwartete Ausgabe:**
```
[20, 50, 50, 100, 200, 200, 300, 500, 700, 1000, 1000, 2000] 6120
```

In [ ]:
def lies_stuecke(dateiname):
    stuecke = []
    # dein Code
    return stuecke

stuecke = lies_stuecke('daten/waage_stuecke.txt')
print(stuecke, sum(stuecke))

<details>
<summary>Tipp anzeigen</summary>

`zahlen = [int(wort) for wort in datei.read().split()]`, danach `zahlen[1:]` (oder bis `1 + zahlen[0]`).

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def lies_stuecke(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        zahlen = [int(wort) for wort in datei.read().split()]
    return zahlen[1:1 + zahlen[0]]

stuecke = lies_stuecke('daten/waage_stuecke.txt')
print(stuecke, sum(stuecke))
```

Zwölf Stücke von 20 g bis 2 000 g, zusammen 6 120 g. Mehr als 6 120 g kann man also nie wiegen.

</details>

#### Aufgabe F.2

Schreibe `zaehle_wiegbar(stuecke, ziele)`: Sie teilt die Stücke in zwei Hälften, berechnet **einmal** die Summen beider Hälften (`alle_summen`, die hinteren als Menge) und zählt, wie viele
der `ziele` genau gewogen werden können. Das Gerüst enthält `alle_summen` und `lies_stuecke`.

**Erwartete Ausgabe:**
```
367
1
```

In [ ]:
def alle_summen(stuecke):
    summen = [0]
    for gewicht in stuecke:
        neu = []
        for s in summen:
            for faktor in (-1, 0, 1):
                neu.append(s + faktor * gewicht)
        summen = neu
    return summen

def lies_stuecke(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        zahlen = [int(wort) for wort in datei.read().split()]
    return zahlen[1:1 + zahlen[0]]

def zaehle_wiegbar(stuecke, ziele):
    h = len(stuecke) // 2
    anzahl = 0
    # dein Code
    return anzahl

stuecke = lies_stuecke('daten/waage_stuecke.txt')
ziele = range(10, 10001, 10)
print(zaehle_wiegbar(stuecke, ziele))
print(zaehle_wiegbar(stuecke, [440, 6120, 6130]))

<details>
<summary>Tipp anzeigen</summary>

Berechne die Summen **vor** der Schleife über die `ziele`, nicht für jedes Ziel neu. Für jedes `ziel`: `for a in summen_vorn:` und `if ziel - a in menge_hinten:` erhöht `anzahl` und beendet die innere Schleife mit `break`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def alle_summen(stuecke):
    summen = [0]
    for gewicht in stuecke:
        neu = []
        for s in summen:
            for faktor in (-1, 0, 1):
                neu.append(s + faktor * gewicht)
        summen = neu
    return summen

def lies_stuecke(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        zahlen = [int(wort) for wort in datei.read().split()]
    return zahlen[1:1 + zahlen[0]]

def zaehle_wiegbar(stuecke, ziele):
    h = len(stuecke) // 2
    summen_vorn = alle_summen(stuecke[:h])
    menge_hinten = set(alle_summen(stuecke[h:]))
    anzahl = 0
    for ziel in ziele:
        for a in summen_vorn:
            if ziel - a in menge_hinten:
                anzahl = anzahl + 1
                break                        # ein Treffer genügt
    return anzahl

stuecke = lies_stuecke('daten/waage_stuecke.txt')
ziele = range(10, 10001, 10)
print(zaehle_wiegbar(stuecke, ziele))
print(zaehle_wiegbar(stuecke, [440, 6120, 6130]))
```

367 von 1 000 Zielen (10 g bis 10 000 g) lassen sich genau wiegen. Von den drei Gewichten der zweiten Zeile ist nur 6 120 g wiegbar (alle Stücke rechts): 440 g nicht, und 6 130 g ist schwerer als alle Stücke zusammen.

</details>

#### Aufgabe F.3

Nicht alle Ziele sind wiegbar. Schreibe `groesste_abweichung(stuecke, ziele)`: Für jedes Ziel soll das Programm wie in 4.4 die **nächstliegende** wiegbare Summe suchen (mit `bisect_left` in den sortierten
hinteren Summen) und die **größte** Abweichung `abs(summe - ziel)` über alle Ziele zurückgeben.

**Erwartete Ausgabe:**
```
10
```

In [ ]:
def alle_summen(stuecke):
    summen = [0]
    for gewicht in stuecke:
        neu = []
        for s in summen:
            for faktor in (-1, 0, 1):
                neu.append(s + faktor * gewicht)
        summen = neu
    return summen

def lies_stuecke(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        zahlen = [int(wort) for wort in datei.read().split()]
    return zahlen[1:1 + zahlen[0]]

from bisect import bisect_left

def groesste_abweichung(stuecke, ziele):
    h = len(stuecke) // 2
    summen_vorn = alle_summen(stuecke[:h])
    hinten_sortiert = sorted(alle_summen(stuecke[h:]))
    groesste = 0
    # dein Code
    return groesste

stuecke = lies_stuecke('daten/waage_stuecke.txt')
print(groesste_abweichung(stuecke, range(10, 6121, 10)))

<details>
<summary>Tipp anzeigen</summary>

Für jedes `ziel` eine Variable `beste = None`. Für jedes `a` der vorderen Summen: `i = bisect_left(hinten_sortiert, ziel - a)` und die Nachbarn `i - 1` und `i` (wenn `0 <= j < len(hinten_sortiert)`)
prüfen. Nach den Summen vergleicht man `beste` mit `groesste`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def alle_summen(stuecke):
    summen = [0]
    for gewicht in stuecke:
        neu = []
        for s in summen:
            for faktor in (-1, 0, 1):
                neu.append(s + faktor * gewicht)
        summen = neu
    return summen

def lies_stuecke(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        zahlen = [int(wort) for wort in datei.read().split()]
    return zahlen[1:1 + zahlen[0]]

from bisect import bisect_left

def groesste_abweichung(stuecke, ziele):
    h = len(stuecke) // 2
    summen_vorn = alle_summen(stuecke[:h])
    hinten_sortiert = sorted(alle_summen(stuecke[h:]))
    groesste = 0
    for ziel in ziele:
        beste = None                                   # kleinste Abweichung für dieses Ziel
        for a in summen_vorn:
            i = bisect_left(hinten_sortiert, ziel - a)
            for j in (i - 1, i):
                if 0 <= j < len(hinten_sortiert):
                    abweichung = abs(a + hinten_sortiert[j] - ziel)
                    if beste is None or abweichung < beste:
                        beste = abweichung
        if beste > groesste:
            groesste = beste
    return groesste

stuecke = lies_stuecke('daten/waage_stuecke.txt')
print(groesste_abweichung(stuecke, range(10, 6121, 10)))
```

Jedes Ziel von 10 g bis 6 120 g liegt höchstens 10 g neben einer wiegbaren Summe. Das Ergebnis wird im Generator mit allen 531 441 Summen geprüft. Das Verfahren gibt auch die Summe selbst her, wenn man `beste` als Paar speichert.

</details>

#### Frage F.4

1. Wie viele Summen berechnet `zaehle_wiegbar` insgesamt, und wie viele wären es ohne Aufteilen? Wie viele Vergleiche macht F.2 für 1 000 Ziele höchstens (nur die Größenordnung)?
2. Warum funktioniert Meet in the Middle hier, obwohl die beiden Hälften zusammen die Ware wiegen? Was muss gelten, damit man in zwei Hälften zerlegen darf?
3. Das Lösungsnotebook zur Aufgabe berechnet statt einer Liste ein Wörterbuch: Es merkt sich zu jeder Summe auch, mit wie vielen Stücken man sie erreicht. Wozu braucht man das, und was ändert sich an
   Meet in the Middle?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Zweimal 729 = 1 458 Summen statt 531 441. Für jedes Ziel höchstens 729 Nachschlagen in der Menge, bei 1 000 Zielen also etwa 729 000 Vergleiche. Ohne Aufteilen müsste man 531 441 Summen aufbauen, das Nachschlagen wäre
   danach schnell, aber der Aufbau und der Speicher sind der Engpass.
2. Die Summe zerfällt in **zwei unabhängige Teile**: Die Entscheidungen für die vorderen Stücke haben keinen Einfluss auf die der hinteren, und das Ergebnis ist einfach `a + b`. Bei Aufgaben, wo sich die Teile
   gegenseitig beeinflussen (zum Beispiel „höchstens 5 Stücke insgesamt“), muss man mehr merken.
3. Die Aufgabe will auch wissen, **welche** Stücke auf welche Seite kommen und mit möglichst **wenigen Stücken**. Dafür speichert das Wörterbuch zu jeder Summe die kleinste Stückzahl und die Verteilung.
   Beim Zusammenfügen addiert man die Stückzahlen der beiden Hälften und nimmt unter allen Paaren `a + b = ziel` das mit der kleinsten Summe.

</details>

## Das hast du gelernt

```python
# Binäre Suche in einer sortierten Liste
links = 0
rechts = len(liste) - 1
while links <= rechts:
    mitte = (links + rechts) // 2
    if liste[mitte] == ziel:
        return mitte
    if liste[mitte] < ziel:
        links = mitte + 1
    else:
        rechts = mitte - 1
return -1

# Fertig in Python: bisect
from bisect import bisect_left, bisect_right
i = bisect_left(liste, x)                          # Einfügestelle vor gleichen Elementen
anzahl = bisect_right(liste, b) - bisect_left(liste, a)    # Elemente mit a <= e <= b

# Zielwert raten: das kleinste x, für das geht(x) gilt (geht ist monoton)
while links < rechts:
    mitte = (links + rechts) // 2
    if geht(mitte):
        rechts = mitte
    else:
        links = mitte + 1

# Meet in the Middle
menge_hinten = set(alle_summen(stuecke[h:]))
for a in alle_summen(stuecke[:h]):
    if ziel - a in menge_hinten:
        ...
```

- **O(log n)**: Jeder Schritt halbiert den Bereich, nach etwa log₂(n) Schritten ist man fertig (1 Million Elemente: 20 Schritte).
- Die binäre Suche braucht eine **sortierte** Liste oder eine **monotone** Prüfung. Sonst wirft sie Lösungen weg, ohne einen Fehler zu melden.
- **Grenzen** mit Sorgfalt: Wird der Bereich jede Runde kleiner (`mitte + 1`, `mitte - 1`)? Ist `rechts` ein gültiger Index (`len(liste) - 1`)? Bei „kleinstes x, das geht“ gilt `rechts = mitte`.
- **Zielwert raten und prüfen** macht aus einer Optimierungsaufgabe viele einfache Ja-Nein-Fragen, die oft ein gieriges Verfahren (Notebook 12) beantwortet.
- **Meet in the Middle** spart die Wurzel der Rechenarbeit: 2 · 3^(n/2) statt 3ⁿ. Mit einer Menge oder `bisect` schlägt man die Partner nach.

### Aufgaben der 1. Runde dazu

Bearbeite eine Aufgabe zuerst selbst und sieh dir das Lösungsnotebook erst danach an.

- [Vollgeladen](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/40_Vollgeladen/vollgeladen_loesung.ipynb) (40. BwInf): Bewertung raten, gierig prüfen, binär suchen
- [Marktwaage](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/40_Marktwaage/marktwaage_loesung.ipynb) (40.): Summen aller Verteilungen, nächstliegendes Gewicht mit `bisect`, Meet in the Middle bei der großen Datei

Damit ist der **Kern** des Lehrgangs fertig: Durchprobieren, Backtracking, Schranken, logisches Schließen, Graphen (Breitensuche, Zustände, Dijkstra), Merken, Tabellen, gierige Verfahren und binäre Suche. Weitere Notebooks
folgen, als Nächstes zu **Reihenfolgen** (topologisches Sortieren) und **Bäumen**.